In [1]:
#import libraries
import pandas as pd
import numpy as np

from scipy.stats import pearsonr, spearmanr

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
#Load production sentiment output and source review metadata

sentiment_path = "../../data/samples/yelp_review_sentiment_final.csv"
sample_path = "../../data/samples/yelp_sentiment_sample.csv"

df_sentiment = pd.read_csv(sentiment_path)
df_sample = pd.read_csv(sample_path)

print("Production sentiment dataset")
print(f"Shape: {df_sentiment.shape}")
print(f"Columns: {df_sentiment.columns.tolist()}")

print("\nSource review sample")
print(f"Shape: {df_sample.shape}")
print(f"Columns: {df_sample.columns.tolist()}")

Production sentiment dataset
Shape: (2193, 4)
Columns: ['review_id', 'sentiment_label', 'sentiment_score', 'sentiment_reason']

Source review sample
Shape: (2193, 11)
Columns: ['review_id', 'user_id', 'business_id', 'stars', 'useful', 'funny', 'cool', 'text', 'date', 'period', 'merchant_status']


In [3]:
# Metadata required for validation and downstream analysis
metadata_cols = [
    "review_id",
    "stars",
    "business_id",
    "period",
    "merchant_status"
]

df_metadata = df_sample[metadata_cols].copy()

# Merge sentiment results back to source review metadata
df_validation = df_sentiment.merge(
    df_metadata,
    on="review_id",
    how="left",
    validate="one_to_one"
)

print("Merged validation dataset")
print(f"Shape: {df_validation.shape}")
print(f"Columns: {df_validation.columns.tolist()}")

Merged validation dataset
Shape: (2193, 8)
Columns: ['review_id', 'sentiment_label', 'sentiment_score', 'sentiment_reason', 'stars', 'business_id', 'period', 'merchant_status']


In [4]:
#Merge integrity checks

print("\nMerge Integrity Checks")
print("-" * 40)

print(f"Sentiment records:       {len(df_sentiment):,}")
print(f"Source review records:   {len(df_sample):,}")
print(f"Merged records:          {len(df_validation):,}")
print(f"Unique review IDs:       {df_validation['review_id'].nunique():,}")

# Expected full coverage
assert len(df_sentiment) == 2193, \
    "Expected 2,193 sentiment records."

assert len(df_sample) == 2193, \
    "Expected 2,193 source review records."

assert len(df_validation) == 2193, \
    "Merge changed the expected row count."

assert df_sentiment["review_id"].nunique() == 2193, \
    "Duplicate review IDs in sentiment output."

assert df_sample["review_id"].nunique() == 2193, \
    "Duplicate review IDs in source sample."

assert df_validation["review_id"].nunique() == 2193, \
    "Duplicate review IDs after merge."

# Check that all sentiment records found matching metadata
for col in ["stars", "business_id", "period", "merchant_status"]:
    missing = df_validation[col].isna().sum()
    print(f"Missing {col}: {missing}")

    assert missing == 0, \
        f"Missing source metadata in column: {col}"

print("\n✓ validation passed.")


Merge Integrity Checks
----------------------------------------
Sentiment records:       2,193
Source review records:   2,193
Merged records:          2,193
Unique review IDs:       2,193
Missing stars: 0
Missing business_id: 0
Missing period: 0
Missing merchant_status: 0

✓ validation passed.


**Pearson Correlation: Sentiment Score vs. Star Rating**

Pearson correlation measures the **linear relationship** between the GenAI `sentiment_score` and Yelp `stars` rating.

The analysis uses all **2,193 reviews**. A positive correlation is expected because higher star ratings should generally align with more positive sentiment scores.

The **Pearson correlation coefficient (`r`)** and its **p-value** are calculated using `sentiment_score` and `stars`.


In [5]:
#Pearson correlation

pearson_r, pearson_p = pearsonr(
    df_validation["sentiment_score"],
    df_validation["stars"]
)

print("Pearson Correlation")
print("-" * 40)
print(f"Correlation (r): {pearson_r:.4f}")
print(f"P-value:         {pearson_p:.4e}")

Pearson Correlation
----------------------------------------
Correlation (r): 0.9191
P-value:         0.0000e+00


**Spearman Correlation: Sentiment Score vs. Star Rating**

Spearman correlation measures the **strength and direction of the monotonic relationship** between `sentiment_score` and Yelp `stars`.

Unlike Pearson correlation, it uses **ranked values** and does not require the relationship to be strictly linear.

The **Spearman correlation coefficient (`rho`)** and its **p-value** are calculated using all **2,193 reviews**.

In [6]:
#Spearman correlation

spearman_rho, spearman_p = spearmanr(
    df_validation["sentiment_score"],
    df_validation["stars"]
)

print("Spearman Correlation")
print("-" * 40)
print(f"Correlation (rho): {spearman_rho:.4f}")
print(f"P-value:           {spearman_p:.4e}")

Spearman Correlation
----------------------------------------
Correlation (rho): 0.8221
P-value:           0.0000e+00


### Comparison with the Partial Sentiment Run

| Metric               | Partial Run | Full 2,193 Reviews |  Change |
| -------------------- | ----------: | -----------------: | ------: |
| Pearson correlation  |      0.9270 |             0.9191 | -0.0079 |
| Spearman correlation |      0.8320 |             0.8221 | -0.0099 |

Both correlations decreased slightly in the full dataset but remained **strongly positive**, indicating that completing the sentiment extraction did not materially change the overall relationship between star ratings and GenAI sentiment scores.

The corresponding p-values are effectively zero at the displayed precision, indicating **statistically significant associations, not causation**.

### Star Rating and Sentiment Disagreement Cases

Now we will review the disagreement cases where:

* **5-star reviews are classified as negative**
* **1–2-star reviews are classified as positive**

The original review text is examined to determine whether these cases reflect **mixed reviews, genuine inconsistencies, or differences between the overall rating and sentiment toward a specific aspect of the experience**.


In [7]:
#Identify disagreement cases

# Bring review text into the validation dataframe
df_text = df_sample[["review_id", "text"]].copy()

df_validation = df_validation.merge(
    df_text,
    on="review_id",
    how="left",
    validate="one_to_one"
)

# 5-star reviews classified as negative
five_star_negative = df_validation[
    (df_validation["stars"] == 5) &
    (df_validation["sentiment_label"].str.lower() == "negative")
].copy()

# 1–2-star reviews classified as positive
low_star_positive = df_validation[
    (df_validation["stars"].isin([1, 2])) &
    (df_validation["sentiment_label"].str.lower() == "positive")
].copy()

print("Disagreement cases")
print("-" * 40)
print(f"5-star + negative sentiment:     {len(five_star_negative)}")
print(f"1-2-star + positive sentiment:   {len(low_star_positive)}")
print(f"Total disagreement cases:        "
      f"{len(five_star_negative) + len(low_star_positive)}")

Disagreement cases
----------------------------------------
5-star + negative sentiment:     6
1-2-star + positive sentiment:   6
Total disagreement cases:        12


In [8]:
# Display 5-star / negative cases

print("\n5-STAR REVIEWS CLASSIFIED AS NEGATIVE")
print("=" * 80)

display(
    five_star_negative[
        [
            "review_id",
            "stars",
            "sentiment_label",
            "sentiment_score",
            "sentiment_reason",
            "text"
        ]
    ].head(10)
)


5-STAR REVIEWS CLASSIFIED AS NEGATIVE


,review_id,stars,sentiment_label,sentiment_score,sentiment_reason,text
244,baMyVXVP72Rx6lpfeDhxiQ,5,negative,-0.6,Negative health warning despite recommendation.,Their plant-based Alfredo sauce is indistingui...
284,j0xXeuD8qBvv7hSWu6gtPg,5,negative,-0.5,Negative delivery experience despite good food.,Always been our fall back restaurant for Chine...
645,hSxxSngKzlz3SpPsNPl3lQ,5,negative,-0.5,Criticizes language barrier and ordering exper...,Wanted Caribbean food but just got tired of th...
1103,drnar5HHqDtDauxeVN55zQ,5,negative,-0.5,Several complaints about food quality and salt...,I think this is one of the nicest locations wi...
1958,nOzloVm-3GzAMvcuNAIXIw,5,negative,-0.7,"Positive opening, but strong negative criticis...",Fink's is amazing clean and friendly. A closed...
2065,uKztBO5G5eyFBYlhENSy8A,5,negative,-0.4,"Good concept, poor execution, limited quality.",Great concept but needed better execution. Sh...


In [9]:
# Display 1–2-star / positive cases

print("\n1-2-STAR REVIEWS CLASSIFIED AS POSITIVE")
print("=" * 80)

display(
    low_star_positive[
        [
            "review_id",
            "stars",
            "sentiment_label",
            "sentiment_score",
            "sentiment_reason",
            "text"
        ]
    ].head(10)
)


1-2-STAR REVIEWS CLASSIFIED AS POSITIVE


,review_id,stars,sentiment_label,sentiment_score,sentiment_reason,text
71,rMOr81aMr8a_5_TM-KUj0g,2,positive,0.8,Positive review of food and friendly staff.,Love their meatballs and gravy - best I have ...
629,3t5wTxJXKOSUnSLYIhqZEA,1,positive,0.6,Enthusiastic praise for Capital One coffee sta...,Nothing will make my eyes roll in their socket...
791,UKQNsz6bUJGfDSfptF9kWw,2,positive,0.8,Positive comments about cocktails and familiar...,Before reviewing it's fair to say that the Bla...
1580,1B7QcYn2DjM8MkOD35jfAA,1,positive,0.8,Positive experience and enjoyed the hot chocol...,Maybe it's just best to come here during the d...
1964,LkTpaaNCIBjWOWqhkqOudQ,1,positive,0.4,"Consistent positive experience, friendly staff.",So we normally order pizza from Marcos on Gsnd...
1970,YWUWGtEQ63SBo7iZVXQPJw,2,positive,0.6,"Positive about food, negative about availability.",The jerk wings/sauce and chicken foot soup are...


In [10]:
#Retrieve full text for selected disagreement examples

selected_ids = [
    "baMyVXVP72Rx6lpfeDhxiQ",
    "j0xXeuD8qBvv7hSWu6gtPg",
    "YWUWGtEQ63SBo7iZVXQPJw",
    "1B7QcYn2DjM8MkOD35jfAA"
]

selected_disagreements = df_validation[
    df_validation["review_id"].isin(selected_ids)
].copy()

selected_disagreements[
    [
        "review_id",
        "stars",
        "sentiment_label",
        "sentiment_score",
        "sentiment_reason",
        "text"
    ]
].to_string(index=False)

"             review_id  stars sentiment_label  sentiment_score                                                          sentiment_reason                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

### Star Rating and Sentiment Disagreement Cases

The full validation dataset contains **12 strong star-rating/sentiment disagreement cases**:

* **6** reviews with a 5-star rating but negative sentiment.
* **6** reviews with a 1–2-star rating but positive sentiment.

Four representative cases are presented below for qualitative inspection.

#### Case 1 — 5 Stars, Negative Sentiment

**Review ID:** `baMyVXVP72Rx6lpfeDhxiQ`
**Star Rating:** 5
**Sentiment:** Negative
**Sentiment Score:** -0.6
**Model Reason:** Negative health warning despite recommendation.

> Their plant-based Alfredo sauce is indistinguishable from the traditional kind that will choke off all the blood flow to your heart, killing you to death. Highly recommended.

**Observation:** The review uses strongly negative wording but ends with a clear recommendation. The language appears humorous or sarcastic, showing how literal negative wording can differ from the overall intent of a review.

---

#### Case 2 — 5 Stars, Negative Sentiment

**Review ID:** `j0xXeuD8qBvv7hSWu6gtPg`
**Star Rating:** 5
**Sentiment:** Negative
**Sentiment Score:** -0.5
**Model Reason:** Negative delivery experience despite good food.

> Always been our fall back restaurant for Chinese dining but the funny thing is they do more Vietnamese food. I can't really complain about anything except the delivery service we ordered through for our food really screwed up. We ordered food through them, 30 minutes later they canceled my order and refunded my money. So we called the restaurant and complained...staff was nice and took our order. Ten minutes later the door bell rings and it's the food from the canceled order. Wow! Surprised and hungry we sat down to enjoy and 10 minutes later the doorbell rings again and it's our second order!!!! Two for the price of one???? Tried to explain but the driver smiled and kept saying OK, OK. Haven't ordered through a service with them again.

**Observation:** The reviewer describes a positive relationship with the restaurant but reports a clearly negative delivery experience. This represents a genuine mixed-review case.

---

#### Case 3 — 2 Stars, Positive Sentiment

**Review ID:** `YWUWGtEQ63SBo7iZVXQPJw`
**Star Rating:** 2
**Sentiment:** Positive
**Sentiment Score:** 0.6
**Model Reason:** Positive about food, negative about availability.

> The jerk wings/sauce and chicken foot soup are great when they have it. They never keep it regularly.

**Observation:** The reviewer explicitly praises the food while criticizing its inconsistent availability. The low star rating therefore reflects the overall experience more negatively than the sentiment toward the specific food items.

---

#### Case 4 — 1 Star, Positive Sentiment

**Review ID:** `1B7QcYn2DjM8MkOD35jfAA`
**Star Rating:** 1
**Sentiment:** Positive
**Sentiment Score:** 0.8
**Model Reason:** Positive experience and enjoyed the hot chocolate and deep fried cannoli.

> Maybe it's just best to come here during the day, lol. Loved this place when I first tried it and came back. On my second visit though it wasn't good. Ordered the dark and stormy and it tasted like a extra sweet flat soda so wack and the manager refused to take it back, so either I force myself to drink it or it's a waste. Which is bad business ethics. My wings were soggy. I ordered hot wings and they tasted like they were below mild. It was over all just a wack experience. I liked them the first time but after going to 2 locations and being disappointed in the same night I'm good. Its basically robbery. I explained I didn't like the drink I wasn't intending on drinking the drink and I still have to pay for it and they didn't care that my wings were soggy. I've never been to a restaurant where customers were really unsatisfied and they didn't do anything to try to fix it. What a loser. Now I get why their reviews are so bad. This place ain't gonna make it much longer with these business ethics and I don't care attitude. Unless this place is in a area where quality doesn't matter and they are use to being treated this way. Enjoy the $63!!

**Observation:** The review mentions a positive earlier experience, but the text describing the later visit is overwhelmingly negative. This is therefore a **potentially questionable classification** rather than a straightforward mixed-sentiment case.

### Qualitative Takeaway

The disagreement cases show that **star ratings and review-text sentiment capture related but not identical information**. Ratings reflect the reviewer's overall assessment, while sentiment can be driven by specific aspects of the experience, mixed opinions, humor, or sarcasm. A small number of potentially misclassified cases also suggest that GenAI sentiment should be treated as a **complementary text-based signal rather than a replacement for the original Yelp rating**.



In [11]:
#Aggregate review-level sentiment to merchant level

merchant_sentiment = (
    df_validation
    .groupby("business_id")
    .agg(
        sentiment_score=("sentiment_score", "mean"),
        sentiment_review_count=("review_id", "count")
    )
    .reset_index()
)

# Rename merchant-level sentiment score for clarity
merchant_sentiment = merchant_sentiment.rename(
    columns={"sentiment_score": "merchant_sentiment_score"}
)

print("Merchant-level sentiment")
print("-" * 40)
print(f"Merchants with sentiment: {len(merchant_sentiment):,}")
print(f"Columns: {merchant_sentiment.columns.tolist()}")

print("\nPreview:")
display(merchant_sentiment.head())

Merchant-level sentiment
----------------------------------------
Merchants with sentiment: 300
Columns: ['business_id', 'merchant_sentiment_score', 'sentiment_review_count']

Preview:


,business_id,merchant_sentiment_score,sentiment_review_count
0,-361Hc0tlxSYdrH_C3OgzA,0.605556,9
1,-AWclhh1_2VnqPylPgBU3g,-0.286000,10
2,-SCIBOTjKGJXmeMMPi-uoQ,0.810000,5
3,-ib2qJmDKJgH_ZbtSu_HjA,0.550000,6
4,-n_beuzQuajaezPD2PyQvA,-0.750000,2


In [12]:
# Validate merchant-level aggregation

print("\nAggregation checks")
print("-" * 40)

print(
    f"Review-level sentiment records: "
    f"{len(df_validation):,}"
)

print(
    f"Merchant-level sentiment records: "
    f"{len(merchant_sentiment):,}"
)

print(
    f"Total sentiment reviews represented: "
    f"{merchant_sentiment['sentiment_review_count'].sum():,}"
)

assert merchant_sentiment["sentiment_review_count"].sum() == len(df_validation), \
    "Merchant review counts do not reconcile with review-level records."

assert merchant_sentiment["business_id"].nunique() == len(merchant_sentiment), \
    "Duplicate business IDs found."

assert merchant_sentiment["merchant_sentiment_score"].between(-1, 1).all(), \
    "Merchant sentiment score outside [-1, 1]."

print("\n✓ aggregation checks passed.")


Aggregation checks
----------------------------------------
Review-level sentiment records: 2,193
Merchant-level sentiment records: 300
Total sentiment reviews represented: 2,193

✓ aggregation checks passed.


In [13]:
#Merge merchant sentiment with full engagement dataset

engagement_path = "../../data/samples/yelp_merchant_engagement.csv"

df_engagement = pd.read_csv(engagement_path)

print("Merchant engagement dataset")
print("-" * 40)
print(f"Shape: {df_engagement.shape}")
print(f"Unique merchants: {df_engagement['business_id'].nunique():,}")

Merchant engagement dataset
----------------------------------------
Shape: (8812, 34)
Unique merchants: 8,812


In [14]:
# Merge merchant-level sentiment onto the full engagement dataset

df_merchant_combined = df_engagement.merge(
    merchant_sentiment,
    on="business_id",
    how="left",
    validate="one_to_one"
)

print("\nCombined merchant dataset")
print("-" * 40)
print(f"Shape: {df_merchant_combined.shape}")
print(f"Unique merchants: {df_merchant_combined['business_id'].nunique():,}")


Combined merchant dataset
----------------------------------------
Shape: (8812, 36)
Unique merchants: 8,812


In [15]:
#Validate the merge

print("\nMerge validation")
print("-" * 40)

print(f"Original engagement rows: {len(df_engagement):,}")
print(f"Combined rows:            {len(df_merchant_combined):,}")

print(
    f"Merchants with sentiment: "
    f"{df_merchant_combined['merchant_sentiment_score'].notna().sum():,}"
)

print(
    f"Merchants without sentiment: "
    f"{df_merchant_combined['merchant_sentiment_score'].isna().sum():,}"
)

# Row count must remain unchanged
assert len(df_merchant_combined) == len(df_engagement), \
    "Merchant merge changed the total row count."

# All original merchants must remain
assert (
    df_merchant_combined["business_id"].nunique()
    == df_engagement["business_id"].nunique()
), "Some merchants were lost or duplicated."

# Engagement score must remain unchanged
assert np.allclose(
    df_merchant_combined["engagement_score"],
    df_engagement["engagement_score"],
    equal_nan=True
), "engagement_score was altered during the merge."

# Sentiment score must remain within its expected range
sentiment_present = df_merchant_combined["merchant_sentiment_score"].notna()

assert df_merchant_combined.loc[
    sentiment_present,
    "merchant_sentiment_score"
].between(-1, 1).all(), \
    "Merchant sentiment score outside [-1, 1]."

print("\n✓ merge validation passed.")


Merge validation
----------------------------------------
Original engagement rows: 8,812
Combined rows:            8,812
Merchants with sentiment: 300
Merchants without sentiment: 8,512

✓ merge validation passed.


### Handling Merchants Without Sentiment Coverage

Sentiment-based analysis is restricted to merchants with observed GenAI sentiment coverage.

Of the **8,812 merchants**, **300** have observed sentiment scores, while **8,512** have no sentiment observations. Merchants without coverage retain `NaN` for sentiment fields and are excluded from analyses requiring sentiment.

Missing sentiment is **not treated as neutral sentiment**: `0` represents observed neutral sentiment, while `NaN` indicates that sentiment was not measured.

The `engagement_score` and merchant health classification remain available for **all 8,812 merchants**; only sentiment-dependent analyses are limited to merchants with observed coverage.

In [16]:
#Validate sentiment coverage handling

sentiment_covered = df_merchant_combined[
    df_merchant_combined["merchant_sentiment_score"].notna()
].copy()

sentiment_uncovered = df_merchant_combined[
    df_merchant_combined["merchant_sentiment_score"].isna()
].copy()

print("Sentiment coverage")
print("-" * 40)
print(f"Total merchants:             {len(df_merchant_combined):,}")
print(f"Merchants with sentiment:    {len(sentiment_covered):,}")
print(f"Merchants without sentiment: {len(sentiment_uncovered):,}")

coverage_pct = (
    len(sentiment_covered) / len(df_merchant_combined) * 100
)

print(f"Overall sentiment coverage:  {coverage_pct:.2f}%")

# Verify the expected counts
assert len(sentiment_covered) == 300
assert len(sentiment_uncovered) == 8512
assert len(sentiment_covered) + len(sentiment_uncovered) == 8812

# Confirm missing sentiment remains NaN rather than being replaced with zero
assert (
    df_merchant_combined.loc[
        df_merchant_combined["merchant_sentiment_score"].isna(),
        "merchant_sentiment_score"
    ].isna().all()
)

print("\n✓ validation passed.")

Sentiment coverage
----------------------------------------
Total merchants:             8,812
Merchants with sentiment:    300
Merchants without sentiment: 8,512
Overall sentiment coverage:  3.40%

✓ validation passed.


#### Engagement × Sentiment Polarity 2×2 View

For merchants with observed sentiment coverage, engagement and sentiment are combined into a simple **2×2 decision framework**.

* **High Engagement:** `Growing` or `Stable`
* **Low Engagement:** `Declining`
* **Positive Sentiment:** `merchant_sentiment_score > 0`
* **Negative Sentiment:** `merchant_sentiment_score <= 0`

| Engagement | Sentiment | Business Group          |
| ---------- | --------- | ----------------------- |
| High       | Positive  | **Expand**              |
| High       | Negative  | **Monitor / Intervene** |
| Low        | Positive  | **Growth Opportunity**  |
| Low        | Negative  | **Reassess**            |

Merchants without sentiment observations are excluded from this framework. The four groups form the basis for the subsequent merchant investment priorities.


In [17]:
#Create engagement and sentiment polarity dimensions

# Work only with merchants that have observed sentiment
df_sentiment_merchants = df_merchant_combined[
    df_merchant_combined["merchant_sentiment_score"].notna()
].copy()

# Engagement polarity
df_sentiment_merchants["engagement_polarity"] = np.where(
    df_sentiment_merchants["merchant_status"].isin(["Growing", "Stable"]),
    "High",
    "Low"
)

# Sentiment polarity
df_sentiment_merchants["sentiment_polarity"] = np.where(
    df_sentiment_merchants["merchant_sentiment_score"] > 0,
    "Positive",
    "Negative"
)

print("Engagement X Sentiment dimensions")
print("-" * 40)

print("Engagement polarity:")
print(df_sentiment_merchants["engagement_polarity"].value_counts())

print("\nSentiment polarity:")
print(df_sentiment_merchants["sentiment_polarity"].value_counts())

Engagement X Sentiment dimensions
----------------------------------------
Engagement polarity:
engagement_polarity
High    200
Low     100
Name: count, dtype: int64

Sentiment polarity:
sentiment_polarity
Positive    226
Negative     74
Name: count, dtype: int64


In [18]:
# 2×2 engagement × sentiment view

engagement_sentiment_2x2 = pd.crosstab(
    df_sentiment_merchants["engagement_polarity"],
    df_sentiment_merchants["sentiment_polarity"]
)

# Keep a consistent order
engagement_sentiment_2x2 = engagement_sentiment_2x2.reindex(
    index=["High", "Low"],
    columns=["Positive", "Negative"],
    fill_value=0
)

print("\nEngagement X Sentiment 2x2")
print("-" * 40)

display(engagement_sentiment_2x2)


Engagement X Sentiment 2x2
----------------------------------------


sentiment_polarity,Positive,Negative
engagement_polarity,,
High,147,53
Low,79,21


In [19]:
# Validate the 2×2 framework

total_2x2 = engagement_sentiment_2x2.to_numpy().sum()

print(f"Merchants included in 2x2: {total_2x2:,}")
print(f"Merchants with sentiment:  {len(df_sentiment_merchants):,}")

assert total_2x2 == len(df_sentiment_merchants), \
    "2x2 counts do not reconcile with sentiment-covered merchants."

assert total_2x2 == 300, \
    "Expected all 300 sentiment-covered merchants in the 2x2 framework."

print("\n✓ validation passed.")

Merchants included in 2x2: 300
Merchants with sentiment:  300

✓ validation passed.


### Merchant Investment Priorities

#### Engagement and Sentiment Thresholds

The investment-priority framework uses the **existing engagement classification** and the **binary sentiment threshold** defined above:

* **High Engagement:** `Growing` or `Stable`
* **Low Engagement:** `Declining`
* **Positive Sentiment:** `merchant_sentiment_score > 0`
* **Negative Sentiment:** `merchant_sentiment_score <= 0`

These thresholds are applied only to merchants with observed sentiment coverage and maintain consistency with the engagement × sentiment framework.

A sentiment score of exactly **0** is classified as negative to maintain a mutually exclusive binary classification.


In [20]:
#Validate engagement threshold mapping

print("Engagement classification")
print("-" * 40)

print(
    df_sentiment_merchants[
        ["merchant_status", "engagement_polarity"]
    ].drop_duplicates()
    .sort_values(["merchant_status"])
    .to_string(index=False)
)

print("\nEngagement polarity counts:")
print(df_sentiment_merchants["engagement_polarity"].value_counts())

Engagement classification
----------------------------------------
merchant_status engagement_polarity
      Declining                 Low
        Growing                High
         Stable                High

Engagement polarity counts:
engagement_polarity
High    200
Low     100
Name: count, dtype: int64


In [21]:
#Validate sentiment threshold

print("Sentiment polarity classification")
print("-" * 40)

print(
    df_sentiment_merchants["sentiment_polarity"]
    .value_counts()
)

print("\nThreshold:")
print("Positive: sentiment score > 0")
print("Negative: sentiment score <= 0")

Sentiment polarity classification
----------------------------------------
sentiment_polarity
Positive    226
Negative     74
Name: count, dtype: int64

Threshold:
Positive: sentiment score > 0
Negative: sentiment score <= 0


In [22]:
# Validate sentiment polarity against the numerical threshold

expected_polarity = np.where(
    df_sentiment_merchants["merchant_sentiment_score"] > 0,
    "Positive",
    "Negative"
)

assert (
    df_sentiment_merchants["sentiment_polarity"].values
    == expected_polarity
).all(), "Sentiment polarity does not match the defined threshold."

print("\n✓ validation passed.")


✓ validation passed.


### Assign Merchant Investment Priority Groups

Each sentiment-covered merchant is assigned to one of four investment-priority groups based on its engagement and sentiment classification:

| Group                   | Criteria                             | Interpretation                                                                       |
| ----------------------- | ------------------------------------ | ------------------------------------------------------------------------------------ |
| **Expand**              | High engagement + Positive sentiment | Strong engagement and favorable sentiment; suitable for continued investment.        |
| **Monitor / Intervene** | High engagement + Negative sentiment | Strong engagement but unfavorable sentiment; investigate customer-experience issues. |
| **Growth Opportunity**  | Low engagement + Positive sentiment  | Favorable sentiment with lower engagement; opportunity to increase engagement.       |
| **Reassess**            | Low engagement + Negative sentiment  | Low engagement and unfavorable sentiment; evaluate before additional investment.     |

The classification is applied only to merchants with observed sentiment coverage.


In [23]:
#Assign final priority groups

def assign_priority_group(row):
    if (
        row["engagement_polarity"] == "High"
        and row["sentiment_polarity"] == "Positive"
    ):
        return "Expand"

    elif (
        row["engagement_polarity"] == "High"
        and row["sentiment_polarity"] == "Negative"
    ):
        return "Monitor / Intervene"

    elif (
        row["engagement_polarity"] == "Low"
        and row["sentiment_polarity"] == "Positive"
    ):
        return "Growth Opportunity"

    elif (
        row["engagement_polarity"] == "Low"
        and row["sentiment_polarity"] == "Negative"
    ):
        return "Reassess"

    return np.nan


df_sentiment_merchants["priority_group"] = (
    df_sentiment_merchants.apply(assign_priority_group, axis=1)
)

print("Priority group distribution")
print("-" * 40)

priority_counts = (
    df_sentiment_merchants["priority_group"]
    .value_counts()
)

print(priority_counts)

Priority group distribution
----------------------------------------
priority_group
Expand                 147
Growth Opportunity      79
Monitor / Intervene     53
Reassess                21
Name: count, dtype: int64


In [24]:
# Validate priority-group assignment

print("\nPriority assignment validation")
print("-" * 40)

print(f"Total merchants: {len(df_sentiment_merchants):,}")
print(
    f"Assigned priority groups: "
    f"{df_sentiment_merchants['priority_group'].notna().sum():,}"
)
print(
    f"Unassigned: "
    f"{df_sentiment_merchants['priority_group'].isna().sum():,}"
)

assert df_sentiment_merchants["priority_group"].notna().all(), \
    "Some sentiment-covered merchants were not assigned a priority group."

assert priority_counts.sum() == 300, \
    "Priority group counts do not reconcile with sentiment-covered merchants."

assert set(priority_counts.index) == {
    "Expand",
    "Monitor / Intervene",
    "Growth Opportunity",
    "Reassess"
}, "Unexpected priority group found."

print("\n✓ validation passed.")


Priority assignment validation
----------------------------------------
Total merchants: 300
Assigned priority groups: 300
Unassigned: 0

✓ validation passed.


In [25]:
#Compare full and partial priority distributions

partial_percentages = {
    "Expand": 41.14,
    "Growth Opportunity": 27.76,
    "Monitor / Intervene": 25.42,
    "Reassess": 5.69
}

full_counts = (
    df_sentiment_merchants["priority_group"]
    .value_counts()
)

comparison = pd.DataFrame({
    "priority_group": [
        "Expand",
        "Growth Opportunity",
        "Monitor / Intervene",
        "Reassess"
    ]
})

comparison["partial_run_pct"] = comparison["priority_group"].map(
    partial_percentages
)

comparison["full_run_count"] = comparison["priority_group"].map(
    full_counts
)

comparison["full_run_pct"] = (
    comparison["full_run_count"] / len(df_sentiment_merchants) * 100
)

comparison["change_pp"] = (
    comparison["full_run_pct"]
    - comparison["partial_run_pct"]
)

comparison

,priority_group,partial_run_pct,full_run_count,full_run_pct,change_pp
0,Expand,41.14,147,49.000000,7.860000
1,Growth Opportunity,27.76,79,26.333333,-1.426667
2,Monitor / Intervene,25.42,53,17.666667,-7.753333
3,Reassess,5.69,21,7.000000,1.310000


In [26]:
# Validate final distribution

print("Full priority distribution")
print("-" * 40)

for _, row in comparison.iterrows():
    print(
        f"{row['priority_group']:<22} "
        f"{int(row['full_run_count']):>3} merchants | "
        f"{row['full_run_pct']:.2f}%"
    )

print("\n✓ Comparison calculated.")

Full priority distribution
----------------------------------------
Expand                 147 merchants | 49.00%
Growth Opportunity      79 merchants | 26.33%
Monitor / Intervene     53 merchants | 17.67%
Reassess                21 merchants | 7.00%

✓ Comparison calculated.


#### Priority Group Sanity Checks

A qualitative spot-check is used to verify that merchants are assigned consistently with their underlying engagement and sentiment signals.

**2–3 merchants from each priority group** are reviewed:

* **Expand:** High engagement + positive sentiment
* **Monitor / Intervene:** High engagement + negative sentiment
* **Growth Opportunity:** Low engagement + positive sentiment
* **Reassess:** Low engagement + negative sentiment

This is a **qualitative sanity check, not a statistical validation**. Final group assignments remain determined by the predefined engagement and sentiment thresholds.

In [27]:
# Spot-check 3 merchants from each priority group

spot_check_columns = [
    "business_id",
    "name",
    "merchant_status",
    "engagement_score",
    "merchant_sentiment_score",
    "sentiment_review_count",
    "priority_group"
]

priority_order = [
    "Expand",
    "Monitor / Intervene",
    "Growth Opportunity",
    "Reassess"
]

spot_check_parts = []

for group in priority_order:
    group_data = df_sentiment_merchants[
        df_sentiment_merchants["priority_group"] == group
    ]

    sample = group_data.sample(
        n=min(3, len(group_data)),
        random_state=42
    )

    spot_check_parts.append(sample)

spot_check = pd.concat(spot_check_parts)[spot_check_columns]

display(spot_check)

,business_id,name,merchant_status,engagement_score,merchant_sentiment_score,sentiment_review_count,priority_group
7345,o3mCQJcBBMDWzTY-FuUh9w,Thomas P's Sports Bar & Patio,Stable,0.519633,0.572000,10.0,Expand
3000,4EXm70lqkCxZ-WtgNon8sw,Gabriella's Vietnam,Stable,0.514923,0.490000,10.0,Expand
8318,GoYSJ-YY-YwbxdgasHuq-Q,Mixto,Growing,0.529140,0.120000,10.0,Expand
3226,j6dvIfAzbHccoySvaqjT4w,Burger 21 - New Tampa,Stable,0.516891,-0.010000,10.0,Monitor / Intervene
7102,KiwskluQ7tMqZqvuf872vg,4 Rivers Smokehouse,Growing,0.530413,-0.050000,10.0,Monitor / Intervene
8264,nBqvryCdn9N_vU4rdCbnHw,Spread Bagelry,Stable,0.515263,-0.750000,4.0,Monitor / Intervene
3610,Wtr51cNrv1pYyIRg_IawYQ,The Victor Cafe,Declining,0.455045,0.580000,10.0,Growth Opportunity
168,WO5nr-4sjVs506BdCLL0aA,Kuchi Sushi & Hibachi,Declining,0.502508,0.365000,10.0,Growth Opportunity
2780,or_WcjWELDsssk2JruvJcg,Café Lutecia,Declining,0.502289,0.833333,3.0,Growth Opportunity
706,l7b33ubze8Jqw7C4I1CAuA,Embassy Suites by Hilton Tampa Airport Westshore,Declining,0.498118,-0.200000,8.0,Reassess


In [28]:
# Validate spot-check coverage

spot_check_counts = (
    spot_check["priority_group"]
    .value_counts()
    .reindex(priority_order)
)

print("Spot-check counts")
print("-" * 40)
print(spot_check_counts)

assert all(spot_check_counts >= 2), \
    "Each priority group should have at least 2 spot-check examples."

print("\n✓ spot-check selection passed.")

Spot-check counts
----------------------------------------
priority_group
Expand                 3
Monitor / Intervene    3
Growth Opportunity     3
Reassess               3
Name: count, dtype: int64

✓ spot-check selection passed.


### Overall Takeaway

The completed sentiment analysis provides a **text-based customer-experience signal** that complements the existing merchant engagement score.

Combining engagement and sentiment produces four actionable merchant groups—**Expand, Growth Opportunity, Monitor / Intervene, and Reassess**—while keeping the original engagement scoring framework unchanged.

The results should be interpreted as a **decision-support framework**, with sentiment providing additional context rather than replacing the underlying engagement or Yelp rating measures.

In [29]:
#saving it as a csv file
df_sentiment_merchants.to_csv("../../data/samples/yelp_merchant_priority_final.csv", index=False)